# Kernels en Máquinas de Soporte Vectorial (SVM)

## Introducción Teórica

### ¿Qué es un Kernel?

Un kernel es una función que mapea los datos de entrada a un espacio de características de mayor dimensión, permitiendo que un clasificador lineal encuentre fronteras de decisión no lineales en el espacio original.

### El Truco del Kernel

El kernel trick permite operar en un espacio de características de alta dimensión sin calcular explícitamente las coordenadas en ese espacio. Esto se logra reemplazando el producto punto en el espacio original con una función kernel:

$K(x, x') = \langle \phi(x), \phi(x') \rangle$

Donde $\phi$ es el mapeo al espacio de características.

### Kernels Comunes

| Kernel | Fórmula | Parámetros | Características |
|--------|---------|------------|-----------------|
| **Lineal** | $K(x, x') = x \cdot x'$ | - | Simple, rápido, interpretable |
| **Polinomial** | $(\gamma \cdot x \cdot x' + r)^d$ | d (grado), γ, r | Captura interacciones polinómicas |
| **RBF (Gaussiano)** | $\exp(-\gamma \cdot \|x - x'\|^2)$ | γ | Muy flexible, mapeo a espacio infinito |
| **Sigmoide** | $\tanh(\gamma \cdot x \cdot x' + r)$ | γ, r | Similar a redes neuronales |

### Parámetros Clave

- **C**: Controla el trade-off entre margen y error de clasificación
- **γ (gamma)**: Controla la influencia de un solo punto de entrenamiento en el RBF
- **d (grado)**: Grado del kernel polinomial
- **r (coef0)**: Término independiente en kernels polinomial y sigmoide

In [ ]:
# ============================================================================
# IMPORTS Y CONFIGURACIÓN
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.pipeline import make_pipeline
import mlutils
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline
plt.rcParams["axes.grid"] = False

## 1. Dataset: Anillos Concéntricos

Creamos un dataset con dos anillos concéntricos que no son linealmente separables en el espacio original, ideal para demostrar el poder de los kernels.

### 1.1 Generación y Visualización del Dataset

In [ ]:
X, Y = mlutils.load_dataset_disks(500, seed=39)

fig, (ax1) = plt.subplots(1, 1, figsize=(12, 5))

# Dataset original
ax1.scatter(X[:, 0], X[:, 1], c=Y, s=40, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0, alpha=0.8)
ax1.set_title('Anillos Concéntricos\n(No Linealmente Separables)', fontweight='bold')
ax1.set_xlabel('$x_0$')
ax1.set_ylabel('$x_1$')
ax1.set_aspect('equal', adjustable='datalim')
#ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Número de muestras: {len(X)}")
print(f"Distribución de clases: Clase -1 = {sum(Y==-1)}, Clase +1 = {sum(Y==1)}")


#### Visualización en coordenadas polares


In [ ]:
fig, (ax2) = plt.subplots(1, 1, figsize=(12, 5))

r = np.sqrt(X[:, 0]**2 + X[:, 1]**2)
theta = np.arctan2(X[:, 1], X[:, 0])

ax2.scatter(r, theta, c=Y, s=40, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0, alpha=0.8)
ax2.set_xlabel('Radio (r)', fontsize=12)
ax2.set_ylabel('Ángulo (θ)', fontsize=12)
ax2.set_title('Transformación a Coordenadas Polares\n(Separable Linealmente)', fontweight='bold')
#ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Número de muestras: {len(X)}")
print(f"Distribución de clases: Clase -1 = {sum(Y==-1)}, Clase +1 = {sum(Y==1)}")

## 2. Comparación de Kernels

Evaluamos el rendimiento de diferentes kernels en el dataset de anillos concéntricos.

### 2.1 Kernels Básicos


In [ ]:
kernels_basic = {
    'linear': {},
    'sigmoid': {'gamma': 'auto', 'coef0': 0},
    'rbf': {'gamma': 'auto'},
    'poly': {'degree': 6, 'gamma': 'auto'},
}

plot_kernel_comparison(X, Y, kernels_basic, title='Comparación de Kernels en Anillos Concéntricos')
plt.show()

### Observaciones Iniciales

- **Lineal**: Falla completamente, precisión ~50% (aleatorio)
- **Sigmoide**: Mejora ligeramente pero no es suficiente
- **RBF**: Excelente rendimiento, captura perfectamente la estructura
- **Polinomial**: Buen rendimiento pero con fronteras más complejas

## Sigmoide

$$tanh(\gamma \cdot \langle x \;,\; x^´\rangle + r)$$

In [ ]:
#sklearn.svm.SVC?

In [ ]:
clf = SVC(kernel="sigmoid", gamma="auto")
clf.fit(X, Y)

In [ ]:
mlutils.plot_decision_boundary(lambda x: clf.predict(x), X.T, Y.T)
predictions = clf.predict(X)
print ('Accuracy: %d ' % ((np.sum(Y == predictions))/float(Y.size)*100))

## 3. Análisis del Kernel RBF

El kernel RBF es el más popular debido a su flexibilidad. El parámetro γ controla la influencia de cada punto.

$$exp(-\gamma \cdot {\| x \; - \; x^´\|}²)$$

In [ ]:
clf = SVC(kernel="rbf", gamma="auto")
clf.fit(X, Y)

In [ ]:
mlutils.plot_decision_boundary(lambda x: clf.predict(x), X.T, Y.T)
predictions = clf.predict(X)
print ('Accuracy: %d ' % ((np.sum(Y == predictions))/float(Y.size)*100))


### 3.1 Efecto del Parámetro γ en RBF


In [ ]:


gamma_values = [0.01, 0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(5, 1, figsize=(5, 25))

for idx, gamma in enumerate(gamma_values):
    ax = axes[idx]
    
    clf = SVC(kernel='rbf', gamma=gamma, C=1.0, random_state=42)
    clf.fit(X, Y)
    
    # Visualizar
    DecisionBoundaryDisplay.from_estimator(
        clf, X, ax=ax, grid_resolution=200,
        plot_method='contour', colors='k',
        levels=[0], alpha=0.5
    )
    
    ax.scatter(X[:, 0], X[:, 1], c=Y, s=30, cmap=plt.cm.Spectral,
               edgecolors='k', linewidth=0.0, alpha=0.8)
    
    # Vectores de soporte
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=100, linewidth=2, facecolors='none', edgecolors='red')
    
    accuracy = accuracy_score(Y, clf.predict(X))
    n_support = len(clf.support_)
    
    ax.set_title(f'γ = {gamma}\nPrecisión: {accuracy:.3f}\nSV: {n_support}', fontweight='bold')
    ax.set_xlabel('$x_0$')
    ax.set_ylabel('$x_1$')
    ax.set_aspect('equal', adjustable='datalim')

#plt.suptitle('Efecto de γ en el Kernel RBF', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("="*60)
print("EFECTO DE γ EN EL KERNEL RBF")
print("="*60)
print("""
γ pequeño → Kernel amplio → Frontera suave, menos sobreajuste
γ grande → Kernel estrecho → Frontera compleja, mayor sobreajuste

Valores típicos: 0.1 - 10 (depende de la escala de los datos)
""")

## 4. Análisis del Kernel Polinomial

El kernel polinomial tiene parámetros importantes que afectan el rendimiento.
$$(\gamma \cdot \langle x \;,\; x^´\rangle + r)^d$$

In [ ]:
clf = SVC(kernel="poly", degree=6, gamma="auto")
clf.fit(X, Y);

In [ ]:
mlutils.plot_decision_boundary(lambda x: clf.predict(x), X.T, Y.T)
predictions = clf.predict(X)
print ('Accuracy: %d ' % ((np.sum(Y == predictions))/float(Y.size)*100))


### 4.1 Efecto del Grado en el Kernel Polinomial


In [ ]:
degrees = [2, 3, 4, 5, 6, 7]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, degree in enumerate(degrees):
    ax = axes[idx]
    
    clf = SVC(kernel='poly', degree=degree, gamma='auto', coef0=1, C=1.0, random_state=42)
    clf.fit(X, Y)
    
    DecisionBoundaryDisplay.from_estimator(
        clf, X, ax=ax, grid_resolution=200,
        plot_method='contour', colors='k',
        levels=[0], alpha=0.5
    )
    # Vectores de soporte
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=100, linewidth=2, facecolors='none', edgecolors='red')
    
    ax.scatter(X[:, 0], X[:, 1], c=Y, s=30, cmap=plt.cm.Spectral,
               edgecolors='k', linewidth=0.0, alpha=0.8)
    
    accuracy = accuracy_score(Y, clf.predict(X))
    ax.set_title(f'Grado = {degree}\nPrecisión: {accuracy:.3f}', fontweight='bold')
    ax.set_xlabel('$x_0$')
    ax.set_ylabel('$x_1$')
    ax.set_aspect('equal', adjustable='datalim')

plt.suptitle('Efecto del Grado en el Kernel Polinomial', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Validación Cruzada y Selección de Hiperparámetros

La selección adecuada de hiperparámetros es crucial. Usamos GridSearchCV para encontrar la mejor combinación.


### 5.1 División Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_train[:, 0], X_train[:, 1], c=Y_train, s=30, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0)
ax1.set_title(f'Entrenamiento ({len(X_train)} muestras)', fontweight='bold')
ax1.set_xlabel('$x_0$')
ax1.set_ylabel('$x_1$')
ax1.set_aspect('equal', adjustable='datalim')

ax2.scatter(X_test[:, 0], X_test[:, 1], c=Y_test, s=30, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0)
ax2.set_title(f'Prueba ({len(X_test)} muestras)', fontweight='bold')
ax2.set_xlabel('$x_0$')
ax2.set_ylabel('$x_1$')
ax2.set_aspect('equal', adjustable='datalim')

plt.tight_layout()
plt.show()


### 5.2 Búsqueda de Hiperparámetros con GridSearchCV


In [ ]:


from sklearn.model_selection import GridSearchCV

# Definir pipeline con escalado
pipeline = make_pipeline(
    StandardScaler(),
    SVC(random_state=42)
)

# Grid de parámetros para RBF
param_grid_rbf = {
    'svc__kernel': ['rbf'],
    'svc__C': [0.1, 1.0, 10.0, 100.0],
    'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1.0, 10.0]
}

# Grid de parámetros para Polinomial
param_grid_poly = {
    'svc__kernel': ['poly'],
    'svc__C': [0.1, 1.0, 10.0],
    'svc__degree': [2, 3, 4, 5],
    'svc__gamma': ['scale', 'auto', 0.1, 1.0],
    'svc__coef0': [0, 1, 2]
}

print("Realizando búsqueda de hiperparámetros...")

# Búsqueda RBF
grid_search_rbf = GridSearchCV(pipeline, param_grid_rbf, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_rbf.fit(X_train, Y_train)

# Búsqueda Polinomial
grid_search_poly = GridSearchCV(pipeline, param_grid_poly, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_poly.fit(X_train, Y_train)

# Resultados
print("\n" + "="*60)
print("RESULTADOS DE LA BÚSQUEDA DE HIPERPARÁMETROS")
print("="*60)

print("\n--- Mejor RBF ---")
print(f"Parámetros: {grid_search_rbf.best_params_}")
print(f"CV Score: {grid_search_rbf.best_score_:.4f}")
print(f"Test Score: {grid_search_rbf.score(X_test, Y_test):.4f}")

print("\n--- Mejor Polinomial ---")
print(f"Parámetros: {grid_search_poly.best_params_}")
print(f"CV Score: {grid_search_poly.best_score_:.4f}")
print(f"Test Score: {grid_search_poly.score(X_test, Y_test):.4f}")

### 5.3 Visualización del Mejor Modelo


In [ ]:
from IPython.display import Image
Image('./images/confusion_matrix.png',width=500) 

In [ ]:


best_clf = grid_search_rbf.best_estimator_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Entrenamiento
DecisionBoundaryDisplay.from_estimator(
    best_clf, X_train, ax=ax1, grid_resolution=200,
    plot_method='contour', colors='k', levels=[0], alpha=0.5
)
ax1.scatter(X_train[:, 0], X_train[:, 1], c=Y_train, s=30, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0)
ax1.scatter(best_clf.named_steps['svc'].support_vectors_[:, 0],
            best_clf.named_steps['svc'].support_vectors_[:, 1],
            s=100, linewidth=2, facecolors='none', edgecolors='red')
ax1.set_title(f'Entrenamiento\nPrecisión: {accuracy_score(Y_train, best_clf.predict(X_train)):.3f}',
              fontweight='bold')
ax1.set_xlabel('$x_0$')
ax1.set_ylabel('$x_1$')
ax1.set_aspect('equal', adjustable='datalim')

# Prueba
DecisionBoundaryDisplay.from_estimator(
    best_clf, X_test, ax=ax2, grid_resolution=200,
    plot_method='contour', colors='k', levels=[0], alpha=0.5
)
ax2.scatter(X_test[:, 0], X_test[:, 1], c=Y_test, s=30, cmap=plt.cm.Spectral,
            edgecolors='k', linewidth=0.0)
ax2.set_title(f'Prueba\nPrecisión: {accuracy_score(Y_test, best_clf.predict(X_test)):.3f}',
              fontweight='bold')
ax2.set_xlabel('$x_0$')
ax2.set_ylabel('$x_1$')
ax2.set_aspect('equal', adjustable='datalim')

plt.tight_layout()
plt.show()

# Matriz de confusión
Y_pred = best_clf.predict(X_test)
cm = confusion_matrix(Y_test, Y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Clase -1', 'Clase +1'],
            yticklabels=['Clase -1', 'Clase +1'])
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('Matriz de Confusión - Mejor Modelo', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nReporte de Clasificación - Mejor Modelo:")
print(classification_report(Y_test, Y_pred, target_names=['Clase -1', 'Clase +1']))

## 6. Comparación en Múltiples Datasets

Evaluamos el rendimiento de diferentes kernels en varios tipos de datasets.


### 6.1 Datasets de Prueba


In [ ]:


from sklearn.datasets import make_circles, make_moons, make_blobs

datasets = {
    'Anillos': make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42),
    'Lunas': make_moons(n_samples=300, noise=0.1, random_state=42),
    'Blobs': make_blobs(n_samples=300, centers=2, random_state=42, cluster_std=2.0),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, (X_data, Y_data)) in enumerate(datasets.items()):
    ax = axes[idx]
    ax.scatter(X_data[:, 0], X_data[:, 1], c=Y_data, s=30, cmap=plt.cm.Spectral,
               edgecolors='k', linewidth=0.0)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('$x_0$')
    ax.set_ylabel('$x_1$')
    ax.set_aspect('equal', adjustable='datalim')

plt.tight_layout()
plt.show()


### 6.2 Comparación de Kernels en Diferentes Datasets


In [ ]:


kernels = ['linear', 'rbf', 'poly']
kernel_params = {
    'linear': {},
    'rbf': {'gamma': 'scale'},
    'poly': {'degree': 3, 'gamma': 'scale', 'coef0': 1}
}

results = {}

for dataset_name, (X_data, Y_data) in datasets.items():
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_data, Y_data, test_size=0.3, random_state=42
    )
    
    results[dataset_name] = {}
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Dataset: {dataset_name}', fontsize=16, fontweight='bold')
    
    for idx, kernel in enumerate(kernels):
        ax = axes[idx]
        
        clf = make_pipeline(
            StandardScaler(),
            SVC(kernel=kernel, **kernel_params[kernel], random_state=42)
        )
        clf.fit(X_train, Y_train)
        
        # Visualizar
        DecisionBoundaryDisplay.from_estimator(
            clf, X_test, ax=ax, grid_resolution=200,
            plot_method='contour', colors='k',
            levels=[0], alpha=0.5
        )
        ax.scatter(X_test[:, 0], X_test[:, 1], c=Y_test, s=30, cmap=plt.cm.Spectral,
                   edgecolors='k', linewidth=0.0, alpha=0.8)
        
        train_acc = accuracy_score(Y_train, clf.predict(X_train))
        test_acc = accuracy_score(Y_test, clf.predict(X_test))
        
        results[dataset_name][kernel] = {'train': train_acc, 'test': test_acc}
        
        ax.set_title(f'{kernel}\nTrain: {train_acc:.3f}, Test: {test_acc:.3f}',
                     fontweight='bold')
        ax.set_xlabel('$x_0$')
        ax.set_ylabel('$x_1$')
        ax.set_aspect('equal', adjustable='datalim')
    
    plt.tight_layout()
    plt.show()

# Resumen
print("\n" + "="*60)
print("RESUMEN DE RESULTADOS")
print("="*60)
for dataset_name, dataset_results in results.items():
    print(f"\n{dataset_name}:")
    for kernel, scores in dataset_results.items():
        print(f"  {kernel:8} - Test: {scores['test']:.3f}")

## 7. Visualización de la Transformación del Kernel

Una forma de entender los kernels es visualizar cómo transforman los datos.

In [ ]:
X, Y = load_dataset_disks(500, seed=39)


In [ ]:
# ============================================================================
# 7.1 Matriz de Kernel
# ============================================================================

def plot_kernel_matrix(X, Y, kernel='rbf', gamma=1.0):
    """Visualiza la matriz de kernel para un conjunto de datos."""
    from sklearn.metrics.pairwise import rbf_kernel, polynomial_kernel, sigmoid_kernel, linear_kernel
    
    kernel_functions = {
        'linear': linear_kernel,
        'rbf': lambda X, Y: rbf_kernel(X, Y, gamma=gamma),
        'poly': lambda X, Y: polynomial_kernel(X, Y, degree=3, gamma=gamma, coef0=1),
        'sigmoid': lambda X, Y: sigmoid_kernel(X, Y, gamma=gamma, coef0=0)
    }
    
    K = kernel_functions[kernel](X, X)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Matriz de kernel
    im = ax1.imshow(K, cmap='RdBu_r', aspect='auto')
    ax1.set_title(f'Matriz de Kernel {kernel}', fontweight='bold')
    ax1.set_xlabel('Muestras')
    ax1.set_ylabel('Muestras')
    plt.colorbar(im, ax=ax1)
    
    # Distribución de valores del kernel
    K_flat = K[np.triu_indices_from(K, k=1)]
    ax2.hist(K_flat, bins=30, alpha=0.7, color='blue')
    ax2.axvline(np.mean(K_flat), color='red', linestyle='--', label=f'Media: {np.mean(K_flat):.3f}')
    ax2.axvline(np.median(K_flat), color='green', linestyle='--', label=f'Mediana: {np.median(K_flat):.3f}')
    ax2.set_title(f'Distribución de Similitudes ({kernel})', fontweight='bold')
    ax2.set_xlabel('Valor del Kernel')
    ax2.set_ylabel('Frecuencia')
    ax2.legend()
    
    plt.tight_layout()
    return fig, ax1, ax2

# Seleccionar un subconjunto de puntos para visualización
X_subset = X[:50]
Y_subset = Y[:50]

for kernel in ['linear', 'rbf', 'poly', 'sigmoid']:
    plot_kernel_matrix(X_subset, Y_subset, kernel=kernel, gamma=0.5)
    plt.show()

## 8. Guía de Selección de Kernels

### ¿Qué kernel elegir?

| Situación | Kernel Recomendado | Motivo |
|-----------|-------------------|--------|
| Datos linealmente separables | **Lineal** | Simple, rápido, interpretable |
| Datos no lineales, sin estructura clara | **RBF** | El más versátil |
| Datos con estructura polinomial | **Polinomial** | Captura interacciones polinómicas |
| Datos con características textuales | **Lineal** | Alta dimensionalidad |
| Datos con patrones periódicos | **Sigmoide** o **RBF** | Puede capturar periodicidad |

### Buenas Prácticas

1. **Siempre escalar los datos**: SVM es sensible a la escala de características
2. **Comenzar con kernel lineal**: Si funciona, es la mejor opción
3. **Usar validación cruzada**: Para seleccionar C y γ
4. **No usar γ='auto' en producción**: Es mejor usar 'scale' o tunearlo
5. **Monitorear sobreajuste**: Muchos vectores de soporte pueden indicar sobreajuste

## 9. Aplicación Práctica: Dataset de Kaggle

Ejemplo de uso de SVM en un problema real de clasificación de tarjetas de crédito.

Referencia: [Credit Card Dataset - SVM Classification](https://www.kaggle.com/pierra/credit-card-dataset-svm-classification)

In [ ]:
# ============================================================================
# 9.1 Estructura del Pipeline Completo
# ============================================================================

from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Ejemplo de pipeline completo para un problema de clasificación
print("""
Pipeline completo para clasificación con SVM:
--------------------------------------------
1. Separar características numéricas y categóricas
2. Escalar numéricas (StandardScaler)
3. Codificar categóricas (OneHotEncoder)
4. SVM con kernel RBF y validación cruzada

Ejemplo de código:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(), categorical_features)
])

pipeline = make_pipeline(
    preprocessor,
    SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
)

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f'CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')
""")

## 10. Resumen y Conclusiones

### Principales Aprendizajes

1. **Los kernels permiten separación no lineal** mediante mapeo a espacios de mayor dimensión
2. **El kernel RBF es el más versátil** y funciona bien en la mayoría de los casos
3. **La selección de hiperparámetros es crítica** para el rendimiento
4. **El escalado de datos es obligatorio** para obtener buenos resultados
5. **Los vectores de soporte** son los puntos que definen la frontera

### Recomendaciones Finales

- Comenzar con **kernel lineal** para datasets con muchas características
- Usar **RBF** para la mayoría de problemas no lineales
- Usar **GridSearchCV** para encontrar los mejores parámetros
- **Balancear clases** con class_weight cuando sea necesario
- **Monitorear** el número de vectores de soporte como indicador de complejidad

## Referencias

1. [Scikit-learn: SVM Documentation](https://scikit-learn.org/stable/modules/svm.html)
2. [Scikit-learn: SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
3. [Scikit-learn: Kernel Functions](https://scikit-learn.org/stable/modules/metrics.html#kernel-functions)
4. [Stanford CS229: Support Vector Machines](https://cs229.stanford.edu/notes2023fall/cs229-notes3.pdf)
5. [MIT 6.867: Support Vector Machines](https://ocw.mit.edu/courses/6-867-machine-learning-fall-2006/)
6. [Kaggle: Credit Card Dataset SVM Classification](https://www.kaggle.com/pierra/credit-card-dataset-svm-classification)